In [39]:
import os 
from dotenv import load_dotenv
from knowledge_base_manager.core.qna_manager import QnAManager
from knowledge_base_manager.core.knowledge_base_manager import KnowledgeBaseManager
from knowledge_base_manager.types import Category

# TODO: delete these temp imports
from langchain_openai import AzureChatOpenAI

In [34]:
from ..course_v2.functions.agents.human_loop.feedback import hi

ImportError: attempted relative import with no known parent package

<h2>Initialise Variables</h2>

In [44]:
# Initialise all the variables needed 

# Initialise the LLM used in QnAManager
# self.llm = gpt_4o_mini_azure()

# ignore this
# Defines the instance of AzureChatOpenAI class
llm = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    deployment_name=os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME"),
    model_name=os.environ.get("AZURE_OPENAI_DEPLOYMENT_4o_NAME"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    temperature=0,
)

# Define question categories
question_categories = [
    Category(title="ADMIN", description="Questions about deadlines, submission processes, group work policies, lab sites, or assignment logistics.", example_question="Where do I submit the mini project?"),
    Category(title="TECHNICAL", description="Questions about programming errors, technical setup, or software issues.", example_question="How do I resolve this error when installing the library?"),
    Category(title="CONTENT", description="Questions about course material, lecture content, concepts, or explanations of topics.", example_question="Can you explain the concept of dynamic programming again?"),
    Category(title="EVALUATION", description="Questions about grading criteria, marking schemes, or assessment feedback.", example_question="How many marks is the final project worth?"),
    Category(title="RESOURCE", description="Questions requesting additional resources, study materials, or sample solutions.", example_question="Do you have any sample solutions from last year’s exam?"),
    Category(title="UNCATEGORISED", description="Questions that do not clearly fit into any of the above categories.", example_question="I am confused about something but I’m not sure how to explain it."),
    Category(title="IRRELEVANT", description="Questions that are unrelated to the course or inappropriate.", example_question="What’s the best pizza place near campus?")
]


# Initialise QnA Manager 
qna_manager = QnAManager(db_connection_str=os.environ.get("FB_AZURE_COSMOSDB_CONNECTION_STR"),
    db_name = "courseGenie", # <--- change database name here
    collection_name = "qnaDocument", # <--- change collection name here
    llm=llm,
    rephrase_question=True,
    categorise_question=True,
    categories=question_categories)

# Initialise Knowledge base manager 
kb_manager = KnowledgeBaseManager(
    azure_text_embedding_config={
        "azure_deployment": os.environ.get("TEXT_EMBEDDING_MODEL_DEPLOYMENT"),
        "api_key": os.environ.get("AZURE_OPENAI_API_KEY"),
        "endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
        "model": os.environ.get("TEXT_EMBEDDING_MODEL_NAME")
    },
    azure_ai_search_config={
        "endpoint": os.environ.get("FB_AZURE_AI_SEARCH_ENDPOINT"),
        "api_key": os.environ.get("FB_AZURE_AI_SEARCH_API_KEY")
    },
    index_name="coursegenie-qna"
)

/Users/bern/anaconda3/envs/coursegenie/lib/python3.13/site-packages/knowledge_base_manager/core/database_manager.py:17: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  self.client = pymongo.MongoClient(db_connection_str)


<h2>Usage</h2>

In [45]:
# Create the index
kb_manager.create_index()

'coursegenie-qna'

In [58]:
# Similarity search test
user_query = "Who teaches SC1007?"
retrieved_docs = kb_manager.similarity_search(user_query)

context_str=""

for doc in retrieved_docs:
    context_str.append(doc["content"])

# Retrieve context from kb_manager
print("Context_str:" ,context_str)


Context_str: 


In [ ]:
def classify_query(context_str:str, query:str):

    # Form system prompt
    system_prompt = f"""
                        You are given a context and a query. Determine whether the context provides sufficient information to answer the query.

                        If the context is enough to answer the query, respond to the query using the context.

                        If the context is insufficient to answer the query, respond with "QUERY" only.
s
                        Context: {context_str}
                        Query: {query}
                        """

    llm_response = llm.invoke(system_prompt).content

    return llm_response

print(classify_query(context_str, user_query))

QUERY


In [ ]:
if ("QUERY" in classify_query(context_str, user_query)):
    # add to admin applicaiton QnA database
    qna_manager.add_unanswered_question(user_query)
